# M20 · Picking arrivals: STA/LTA triggers, onset times and picks by hand

**Part 2 · week 3 · about 10 h core**

## 한국어 요약

이 모듈에서는 지진파의 도달 시각을 읽는 방법을 배웁니다. 먼저 진짜 답을 알고 있는 합성 기록으로 STA/LTA 비, 두 문턱값을 쓰는 트리거, AIC 초동 시각을 계산하는 함수를 직접 작성하고 ObsPy의 결과와 비교합니다. 이 세 함수를 묶어 대역 통과 필터를 거친 기록에서 P파를 자동으로 피킹하는 함수를 만들고, 영위상 필터와 인과 필터를 썼을 때 피킹 시각이 참값과 얼마나 달라지는지 신호 대 잡음비별로 먼저 예측한 뒤 직접 재어 봅니다. 그다음 2019년 7월 21일 상주 지진(ML 3.9)의 기록에서 진앙 100 km 안의 관측소 6곳 이상을 골라 피킹 도구(E2)로 P파와 S파를 손으로 읽고, 각 피킹의 불확실성과 P파 초동 극성을 함께 기록합니다. 피킹을 마친 뒤에는 여러분이 읽은 초동 극성과 관측소 메타데이터에 적힌 이득의 부호를 방위각 순서로 나란히 놓고, 비슷한 방위각의 이웃 관측소와 비교합니다. 이어서 자동 피킹 함수를 같은 기록에 돌려 자신의 수동 피킹과 비교하고, 설정값 격자에서 오차 통계를 근거로 설정을 골라 그 이유를 설명합니다. 마지막으로 ObsPy에 들어 있는 Baer-Kradolfer 피커(pk_baer)를 설정 그대로 두 번 돌려 결과가 같은지 확인한 뒤, 설정 하나만 바꾸어 무엇이 달라지는지 설명합니다.

## What you will be able to do

- Write an STA/LTA ratio and a two-threshold trigger, and match ObsPy's results on made-up records, including 32-bit integer data
- Time an onset with the Akaike information criterion inside a window around a trigger
- Measure, on made-up onsets whose true time you know, how far zero-phase and causal band-pass filters move automatic picks, and how that depends on the signal-to-noise ratio
- Pick P and S by hand at six or more stations within 100 km, with an uncertainty for every pick and the first-motion polarity of every clear P
- Compare your automatic picks with your own manual picks, report robust error statistics for a grid of settings, and justify the setting you choose

## Before you start

- Modules: M03, M10
- Reading: Trnkoczy (2012), the whole information sheet (STA/LTA settings); Havskov & Ottemöller (2010), the chapter on routine processing, phase reading; Bormann (ed.) (2012), the chapters on seismic phases and on reading onsets and polarities
- Needs: stp
- Key terms (see the glossary): phase pick (위상 피킹, 도달 시각 읽기), arrival time (도달 시각), onset (온셋, 도달 시작점), impulsive onset (급격한 초동 (임펄스형 온셋)), emergent onset (점진적 초동 (서서히 나타나는 온셋)), pick uncertainty (피킹 불확실성), P wave (P파), S wave (S파), signal-to-noise ratio (SNR; 신호 대 잡음비), first-motion polarity (초동 극성), polarity reversal (극성 반전), stage gain (단계 이득), overall sensitivity (전체 감도), azimuth (방위각), characteristic function (특성 함수), moving average (이동 평균), STA/LTA (단기/장기 평균비), trigger (트리거), trigger threshold (트리거 문턱값), false trigger (오트리거, 잘못된 트리거), integer overflow (정수 오버플로), floating-point number (부동소수점 수), change point (변화점), Akaike information criterion (AIC; 아카이케 정보 기준), band-pass filter (대역 통과 필터), causal filter (인과 필터), zero-phase filter (영위상 필터), taper (테이퍼), detrend (추세 제거), sampling rate (표본화율, 샘플링 레이트), S-P time (S-P 시간), epicentral distance (진앙 거리), hypocentral distance (진원 거리), head wave (선두파), widget (위젯), median absolute error (중앙 절대 오차), deterministic computation (결정론적 계산)

## Passing this module

Each task is done when its `st.check(...)` cell prints PASS. The pass criteria are frozen in [criteria/M20.yaml](../../../.sgtl/criteria/M20.yaml).

## AI use

Follow [docs/ai_policy.md](../../../docs/ai_policy.md) and log every AI conversation in `work/ai_log/M20.md`.

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
DATASET = "sangju2019"                    # folder of your STP download (module M03) inside the data folder
KMA_EVENT_ID = "2019009762"               # the ML 3.9 Sangju earthquake of 21 July 2019 (UTC id 20190721020418)
BAND = "HH"                               # channel folder: HH (broadband seismometers)
MAX_EPI_KM = 100.0                        # pick only stations within this epicentral distance (direct P and S)
FMIN_HZ, FMAX_HZ = 1.0, 10.0              # band-pass corners (Hz) for the automatic picker
STA_S, LTA_S = 0.5, 10.0                  # STA and LTA window lengths (s) to start with
THR_ON, THR_OFF = 4.0, 1.5                # trigger thresholds to start with
AIC_PRE_S, AIC_POST_S = 2.0, 0.5          # AIC window: seconds before and after the trigger-on sample
SNR_LEVELS = (5.0, 20.0, 100.0)           # Part 5: signal-to-noise ratios of the made-up onsets
N_MADE_UP = 40                            # Part 5: made-up records per signal-to-noise ratio
PICK_WINDOW_S = (-20.0, 60.0)             # Part 7: part of each record the automatic picker searches (s after origin)
BAER_CHANGE = ("thr1", 10.0)              # Part 8: the one pk_baer setting you change, and its new value
PICKS_CSV = "../picks/M20_manual_picks.csv"   # your manual picks (in work/picks/); M21 reads them again
FIGURE_DIR = "figures"                    # your figure files go here, next to this notebook
RESULT_DIR = "results"                    # your result files go here, next to this notebook

In [ ]:
import sgtl_tutorial as st
S = st.init("M20")

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import UTCDateTime, read, read_inventory
from obspy.geodetics import gps2dist_azimuth

from sgtl_tutorial import synth_onsets

Path(FIGURE_DIR).mkdir(exist_ok=True)
Path(RESULT_DIR).mkdir(exist_ok=True)
Path(PICKS_CSV).parent.mkdir(parents=True, exist_ok=True)

## How this notebook works

The same five steps as in the earlier modules, for every task: read the **Contract**, write a **prediction**,
replace `raise NotImplementedError(...)` under `# YOUR CODE HERE` with your code, run the check cell, and answer
the **Explain** prompt in 3-5 sentences of your own. `st.hint("M20-T1", 1)` shows a first hint; hints 2 and 3
unlock after failed checks. The checks try your functions on made-up records whose answers are known from how
they were built (`st.reroll("M20-T1")` draws new ones) or come from ObsPy, and on your own Sangju files; for
those the check never prints its own values.

The module climbs three steps. First you write the parts of an automatic picker yourself, on made-up records
whose true arrival time you know (Parts 2-5). Then you pick real records by hand (Part 6) and compare your
picker with your own picks (Part 7). Last, you run a picker that ships with ObsPy, unchanged, once more unchanged
and then with one setting changed (Part 8).

You may ask an AI tool to explain an error message or a concept. It may not write the task functions, your
predictions, your picks or your Explain answers ([docs/ai_policy.md](../../../docs/ai_policy.md)); log AI
conversations in `work/ai_log/M20.md`.

## Part 0 · Your data (about 15 min)

The cell below takes the ML 3.9 Sangju earthquake from the KMA catalog that comes with the course (times in
UTC), finds its waveforms in your M03 download, and lists the broadband stations by **epicentral distance**
(glossary `epicentral_distance`, the distance along the surface from the epicenter to the station) and
**hypocentral distance** (glossary `hypocentral_distance`, the straight distance from the hypocenter).

In [ ]:
cat = st.data.kma_catalog()
event = cat.loc[cat["event_id"] == KMA_EVENT_ID].iloc[0]
ORIGIN = UTCDateTime(event["origin_utc"].isoformat())
EV_LAT, EV_LON = float(event["latitude"]), float(event["longitude"])
EV_DEPTH_KM, EV_ML = float(event["depth_km"]), float(event["ml"])
print(f"KMA event {KMA_EVENT_ID}: origin {ORIGIN} (UTC), {EV_LAT:.2f} N, {EV_LON:.2f} E, depth {EV_DEPTH_KM:g} km, "
      f"ML {EV_ML:.1f}")

SAC_DIR = st.data.root() / DATASET / "stp_download" / "SAC" / ORIGIN.strftime("%Y%m%d%H%M%S")
if not (SAC_DIR / BAND).is_dir():
    raise FileNotFoundError(f"There are no {BAND} waveforms of this event in {SAC_DIR}. Fetch the {DATASET} data "
                            "with STP first (module M03), or set DATASET in the parameters cell to your folder.")
listed = []
for z_file in sorted((SAC_DIR / BAND).glob(f"*.{BAND}Z.sac")):
    head = read(str(z_file), headonly=True)[0].stats
    epi_km = gps2dist_azimuth(EV_LAT, EV_LON, head.sac.stla, head.sac.stlo)[0] / 1000.0
    listed.append({"station": f"{head.network}.{head.station}", "epi_km": epi_km,
                   "hypo_km": float(np.hypot(epi_km, EV_DEPTH_KM)), "z_file": z_file})
stations = pd.DataFrame(listed).sort_values("epi_km", ignore_index=True)
print(f"{len(stations)} stations with {BAND}Z records; {int((stations['epi_km'] <= MAX_EPI_KM).sum())} within "
      f"{MAX_EPI_KM:g} km of the epicenter")
print(stations[["station", "epi_km", "hypo_km"]].round(1).to_string(index=False))

NEAREST = stations.loc[0, "station"]
near = read(str(SAC_DIR / BAND / f"*.{NEAREST}.{BAND}?.sac"))
near.sort(keys=["channel"])
near_z = near.select(component="Z")[0]
print(near)

## Part 1 · What a pick is (about 30 min)

The **arrival time** (glossary `arrival_time`) of a wave is the moment it reaches a station. A **phase pick**
(glossary `phase_pick`) is a measured arrival time of one phase at one station: here the **P wave** (glossary
`p_wave`, the first and fastest body wave, best seen on the vertical component) and the **S wave** (glossary
`s_wave`, slower and usually largest on the horizontal components). Locating an earthquake (M31) and finding its
mechanism (M51) start from picks, so every pick carries three things:

- the time of the **onset** (glossary `onset`): the first sample where the new wave shows in the record. An
  **impulsive onset** (glossary `impulsive_onset`) starts within a sample or two; an **emergent onset**
  (glossary `emergent_onset`) grows slowly out of the noise;
- a **pick uncertainty** (glossary `pick_uncertainty`): how far the true onset may lie from your pick, in
  seconds. It is small for an impulsive onset in a quiet record and large for an emergent one in a noisy record;
  Diehl et al. (2009) show how carefully chosen uncertainties make picks from different people agree;
- for P, the **first-motion polarity** (glossary `polarity`): whether the ground first moved up (U, a push away
  from the source) or down (D, a pull toward it), or `?` when you cannot tell.

How clear an onset is depends on the **signal-to-noise ratio** (glossary `signal_to_noise_ratio`, SNR): how much
larger the arrival is than the background noise. The lab rule for location applies here too: use the direct P
and S waves within about 100 km of the epicenter. Farther away, a **head wave** (glossary `head_wave`) that
travels along the base of the crust can arrive first (M30); never pick it for a location.

The figure shows the first 12 s of the nearest station after the origin, then the first second of its P wave.

In [ ]:
shown = near.copy().detrend("demean")
fig, axes = plt.subplots(4, 1, figsize=(7.5, 7.0))
for ax, tr in zip(axes[:3], shown):
    t_rel = tr.times(reftime=ORIGIN)
    keep = (t_rel >= -2.0) & (t_rel <= 12.0)
    ax.plot(t_rel[keep], tr.data[keep], color="k", lw=0.5)
    ax.set_ylabel(f"{tr.stats.channel} (counts)")
axes[2].set_xlabel("Time after origin (s)")
z_rel = shown.select(component="Z")[0]
t_rel = z_rel.times(reftime=ORIGIN)
quiet = (t_rel > -25.0) & (t_rel < -5.0)
noise_counts = float(np.std(z_rel.data[quiet]))
peak_counts = float(np.abs(z_rel.data[(t_rel > 0.0) & (t_rel < 12.0)]).max())
first_big = float(t_rel[np.flatnonzero((np.abs(z_rel.data) > 20.0 * noise_counts) & (t_rel > -5.0))[0]])
zoom = (t_rel > first_big - 0.6) & (t_rel < first_big + 0.4)
axes[3].plot(t_rel[zoom], z_rel.data[zoom], "k.-", lw=0.6, ms=2)
axes[3].set_ylim(-30.0 * noise_counts, 30.0 * noise_counts)
axes[3].set_xlabel("Time after origin (s); every dot is one sample")
axes[3].set_ylabel(f"{z_rel.stats.channel} (counts)")
fig.tight_layout()
fig.savefig(Path(FIGURE_DIR) / "nearest_station_first_seconds.png", dpi=150)
print(f"{NEAREST}: noise {noise_counts:.0f} counts (standard deviation 25-5 s before the origin); largest vertical "
      f"swing in the first 12 s {peak_counts:.0f} counts, about {peak_counts / noise_counts:.0f} times the noise")

In the bottom panel the noise is a thin band and the P wave leaves it within one or two samples: an impulsive
onset with a large SNR. Most earthquakes you will meet are smaller or farther away, and their onsets hide in the
noise. An automatic picker has to find them anyway, and it has to say when it is not sure. Parts 2-5 build one.

## Part 2 · The STA/LTA ratio (about 30 min, then task 1)

Allen (1978) proposed the detector that most networks still use. First turn the record into a
**characteristic function** (glossary `characteristic_function`): a trace whose changes the detector looks for;
here the squared samples, which follow the energy of the ground motion. Then take two **moving averages**
(glossary `moving_average`, the average of the last N samples, recomputed at every sample) of it: a short one,
the STA (short-term average, about the length of an arrival's first cycles), and a long one, the LTA (long-term
average, many seconds of background). Their ratio, **STA/LTA** (glossary `sta_lta`), stays near 1 in steady
noise and jumps when a new arrival comes in, because the short window fills with the arrival long before the
long one does. Trnkoczy (2012) explains how to choose the two window lengths; Withers et al. (1998) compare this
detector with others.

One detail matters in code: a moving average needs a full window. Before the long window is full, the ratio is
not defined, and this course sets it to 0.

### M20-T1 · An STA/LTA ratio · core · about 60 min

**Goal:** write `sta_lta(data, nsta, nlta)`: the STA/LTA ratio of a record, sample by sample.

**Given:** the samples of a record as numbers, and two window lengths in samples.

**Contract:**
- Input: `data`, a 1-D sequence of numbers (a list, or a numpy array of any number type, 32-bit integers
  included); `nsta` and `nlta`, the window lengths in samples (whole numbers, `1 <= nsta < nlta <= len(data)`).
- Output: a 1-D numpy array of 64-bit floats, as long as `data`.
- The characteristic function is the squared samples, `data[j] ** 2`.
- STA at sample `i` is the mean of the squared samples over the `nsta` samples that end at `i` (samples
  `i - nsta + 1` to `i`); LTA at sample `i` is the same over the `nlta` samples that end at `i`. Both windows end
  at the current sample: the ratio looks only backward in time.
- `ratio[i] = STA[i] / LTA[i]` from `i = nlta - 1` on (the first sample where the long window is full);
  `ratio[i] = 0` before that, and also wherever the LTA is 0 (a stretch of zeros).
- Example: `data = [1, 1, 1, 1, 2, 2]`, `nsta = 2`, `nlta = 4` gives `[0, 0, 0, 1.0, 1.4286, 1.6]` (rounded).
- Write the ratio yourself: ObsPy's own STA/LTA functions (`obspy.signal.trigger`, and the `trigger` method of
  a trace or a stream) and the pyrocko package are not allowed here; numpy is.

**Done when** `st.check("M20-T1", func=sta_lta)` passes: on made-up records (noise with arrivals of known size,
stretches of zeros at the start, 32-bit integer and float records with large values), compared with ObsPy's
`classic_sta_lta`.

**Stuck?** `st.hint("M20-T1", 1)`

In [ ]:
# Before you code: noise whose size grows 5 times at sample 3000, with nsta = 50 and nlta = 1000. What is the
# ratio long before the jump, about how large is it 50 samples after the jump, and what is it 1000 samples after
# the jump? Then: what happens if you square 32-bit integer counts of 100,000?
st.predict("M20-T1", """

""")

In [ ]:
def sta_lta(data, nsta, nlta):
    """STA/LTA ratio of the squared samples; both windows end at the current sample.

    data       : 1-D sequence of numbers (any number type)
    nsta, nlta : window lengths in samples, 1 <= nsta < nlta <= len(data)
    returns    : numpy float64 array as long as data; 0 before sample nlta - 1 and wherever the LTA is 0
    """
    # Step 1: a float64 array of the squared samples (the characteristic function).
    # Step 2: the mean over the last nsta and over the last nlta samples, at every sample from nlta - 1 on. A fast
    #         way: keep a running total of the squares; the sum over a window is the difference of two totals.
    # Step 3: divide where the LTA is above 0; leave 0 everywhere else.
    # YOUR CODE HERE
    raise NotImplementedError("M20-T1: write your code here")

In [ ]:
st.check("M20-T1", func=sta_lta)

The check also ran your function on 32-bit records with large values. Compare the second part of your prediction
with three counts squared, once as 32-bit integers and once as 64-bit floats:

In [ ]:
counts_32 = np.array([30_000, 100_000, 300_000], dtype=np.int32)
print("int32 counts squared:  ", counts_32 * counts_32)
print("float64 counts squared:", counts_32.astype(np.float64) ** 2)

Raw counts come as 32-bit numbers. A 32-bit integer holds values up to about 2.1 billion, so a square larger than
that does not fit: the result wraps around to a wrong value, even a negative one, without any warning (an
**integer overflow**, glossary `integer_overflow`). A 32-bit **floating-point number** (glossary
`floating_point_number`) keeps only about 7 digits, too few for long sums of squares. Converting the samples to
64-bit floats (`np.float64`) before squaring them avoids both.

Your function on a made-up record with one arrival at a known time (from `synth_onsets`, the course's generator
of practice records). The noise contains slow swells near 0.2 Hz, like the ocean noise in real records; Part 5
deals with them.

In [ ]:
demo_trace, demo_truth = synth_onsets.onset_trace(np.random.default_rng(20), snr=8.0)
demo_rate = demo_trace.stats.sampling_rate
demo_ratio = sta_lta(demo_trace.data, int(STA_S * demo_rate), int(LTA_S * demo_rate))
t_demo = demo_trace.times()
fig, (ax_rec, ax_ratio) = plt.subplots(2, 1, figsize=(7.5, 4.6), sharex=True)
ax_rec.plot(t_demo, demo_trace.data, color="k", lw=0.5)
ax_rec.set_ylabel("Made-up record")
ax_ratio.plot(t_demo, demo_ratio, color="k", lw=0.8)
ax_ratio.set_ylabel("STA/LTA")
ax_ratio.set_xlabel("Time after the first sample (s); dashed: the true onset")
for ax in (ax_rec, ax_ratio):
    ax.axvline(demo_truth["onset_s"], color="tab:red", lw=0.8, ls="--")
fig.tight_layout()
fig.savefig(Path(FIGURE_DIR) / "sta_lta_made_up.png", dpi=150)
after = t_demo >= demo_truth["onset_s"]
print(f"true onset {demo_truth['onset_s']:.2f} s; largest ratio {demo_ratio.max():.1f} at "
      f"{t_demo[np.argmax(demo_ratio)]:.2f} s; largest ratio before the onset {demo_ratio[~after].max():.2f}")

**Explain (3-5 sentences):** In your figure, where does the ratio begin to rise compared with the true onset,
and why can your ratio never rise before it? How long after the onset does the ratio peak, and what sets that
delay? What would change if the long window ended where the short window begins, so that the two never overlap,
as some programs do? And what became of the 32-bit square of 100,000 counts, compared with your prediction?

## Part 3 · Triggers: on and off (about 20 min, then task 2)

A **trigger** (glossary `trigger`) is the moment a detector declares that an arrival has started. With one
threshold, a ratio that wobbles around it switches on and off many times during one arrival. Detectors therefore
use two **trigger thresholds** (glossary `trigger_threshold`): the trigger turns on when the ratio rises above
`thr_on` and turns off only when it falls back to `thr_off`, a lower value. A trigger on noise, a spike or
another earthquake is a **false trigger** (glossary `false_trigger`).

### M20-T2 · Trigger on and off · core · about 40 min

**Goal:** write `trigger_windows(cft, thr_on, thr_off)`: the stretches of samples where a ratio declares an
arrival.

**Given:** a characteristic function or a ratio, such as your STA/LTA, and two thresholds.

**Contract:**
- Input: `cft`, a 1-D sequence of numbers; `thr_on` and `thr_off`, two numbers with `thr_off < thr_on`.
- Output: a list of `(on, off)` pairs of sample indices (Python `int`), in time order; an empty list when nothing
  triggers.
- A trigger turns on at the first sample whose value is greater than `thr_on`. It stays on while the values stay
  greater than `thr_off`; `off` is the last sample that is still greater than `thr_off`. If the record ends while
  the trigger is on, `off` is the last sample. After `off`, the next trigger needs a value greater than `thr_on`
  again.
- "Greater" is strict: a value equal to `thr_on` does not turn a trigger on, and a value equal to `thr_off` turns
  it off.
- Example: `cft = [0, 5, 5, 2, 1, 0, 5, 0]`, `thr_on = 4`, `thr_off = 1.5` gives `[(1, 3), (6, 6)]`.
- Write it yourself: `obspy.signal.trigger`, the `trigger` method of a trace or a stream, and the pyrocko package
  are not allowed here.

**Done when** `st.check("M20-T2", func=trigger_windows)` passes: on made-up curves (with values exactly at the
thresholds, and triggers still on at the end), compared with ObsPy's `trigger_onset`.

**Stuck?** `st.hint("M20-T2", 1)`

In [ ]:
# Before you code: how many triggers does cft = [0, 5, 3, 5, 1, 5] give with thr_on = 4 and thr_off = 1.5, and
# where? How many with a single threshold of 4 for both on and off? Why do detectors use two thresholds?
st.predict("M20-T2", """

""")

In [ ]:
def trigger_windows(cft, thr_on, thr_off):
    """(on, off) sample pairs where cft first rises above thr_on and stays above thr_off.

    cft             : 1-D sequence of numbers (for example an STA/LTA ratio)
    thr_on, thr_off : the on and off thresholds, thr_off < thr_on
    returns         : list of (on, off) pairs of int sample indices, in time order ([] when nothing triggers)
    """
    # Step 1: walk through the samples, remembering whether a trigger is on and where it started.
    # Step 2: with no trigger on, start one at a value > thr_on; with one on, end it at a value <= thr_off
    #         (its off is the sample before).
    # Step 3: a trigger still on at the end closes at the last sample.
    # YOUR CODE HERE
    raise NotImplementedError("M20-T2: write your code here")

In [ ]:
st.check("M20-T2", func=trigger_windows)

Your two functions on the vertical record of the nearest station, first as recorded and then band-passed
(causal, `FMIN_HZ`-`FMAX_HZ`). The times are seconds after the origin.

In [ ]:
for label, record in (("as recorded", near_z.copy().detrend("demean")),
                      ("band-passed", near_z.copy().detrend("demean").taper(0.05, type="hann", max_length=5.0).filter(
                          "bandpass", freqmin=FMIN_HZ, freqmax=FMAX_HZ, corners=4, zerophase=False))):
    rate = record.stats.sampling_rate
    windows = trigger_windows(sta_lta(record.data, int(STA_S * rate), int(LTA_S * rate)), THR_ON, THR_OFF)
    starts = [round(float(record.times(reftime=ORIGIN)[on]), 2) for on, _ in windows]
    print(f"{label:>12s}: {len(windows)} triggers; on at {starts[:8]} s after the origin"
          + (" ..." if len(starts) > 8 else ""))

**Explain (3-5 sentences):** How many triggers did you get on the record as recorded and after the band-pass,
and which one is the P wave? What would happen to the number of triggers if `thr_on` were 2? The P trigger
comes a little after the onset you saw in Part 1: why must a trigger come late, and what decides by how much?

**Explorer E3 (optional, not checked): an STA/LTA tuner.** A **widget** (glossary `widget`) is a control in a
notebook that runs code again when you move it. The tuner draws the nearest station's vertical record, your own
`sta_lta` ratio and the triggers of your own `trigger_windows`, with sliders for the two windows, the two
thresholds and the band-pass. It never computes a ratio or a trigger itself: without your functions it shows the
record only. Things to try: make the STA window 0.05 s and then 2 s; raise the LTA window from 2 s to 60 s; set
`thr_on` to 2 and to 15; switch the band-pass off. Which setting gives one trigger for the P wave and nothing in
the noise?

In [ ]:
# Optional explorer E3 (not checked): your STA/LTA and triggers on the nearest station's vertical record.
st.widgets.stalta_tuner(near_z, sta_lta=sta_lta, trigger=trigger_windows, reftime=ORIGIN, window=(-25.0, 60.0),
                        fmin=FMIN_HZ, fmax=FMAX_HZ, sta_s=STA_S, lta_s=LTA_S, thr_on=THR_ON, thr_off=THR_OFF,
                        taper=0.01, xlabel="Time after origin (s)")

## Part 4 · Timing an onset: the Akaike information criterion (about 15 min, then task 3)

A trigger says that something arrived, not exactly when. To time the onset, split a short window around the
trigger into two parts, noise before and signal after, and ask where the split fits best. The sample where the
statistics of a record change is a **change point** (glossary `change_point`). Maeda (1985) scores every possible
split with the **Akaike information criterion** (glossary `akaike_information_criterion`, a score that weighs how
well a model fits against how many numbers it needs; you met it in M10): treat each part as noise with its own
variance; a part described well by a small variance scores low. The split with the lowest score is the onset.

### M20-T3 · The AIC onset · core · about 45 min

**Goal:** write `aic_onset(data)`: the first sample of the second part of a short record, by Maeda's AIC.

**Given:** the samples of a short window that holds noise and then the start of an arrival.

**Contract:**
- Input: `data`, a 1-D sequence of at least 5 numbers.
- Output: a Python `int` `k` with `2 <= k <= len(data) - 2`: the index of the first sample of the second part.
- For every `k` from 2 to `N - 2` (`N = len(data)`):
  `AIC(k) = k * ln(var(data[:k])) + (N - k - 1) * ln(var(data[k:]))`, where `var` is the plain variance (numpy's
  `np.var`, which divides by the number of samples) and `ln` the natural logarithm. Return the `k` with the
  smallest AIC; on a tie, the smallest such `k`.
- Example: `data = [1, -1, 1, -1, 1, 6, -6, 6, -6, 6]` gives `5`.
- Write it yourself: `obspy.signal.trigger` (it has an AIC function), the `trigger` method of a trace or a
  stream, and the pyrocko package are not allowed here.

**Done when** `st.check("M20-T3", func=aic_onset)` passes: on records whose noise grows by a known factor at a
known sample, compared with that sample and with ObsPy's `aic_simple`.

**Stuck?** `st.hint("M20-T3", 1)`

In [ ]:
# Before you code: 400 samples of noise that grows 10 times at sample 250. Sketch AIC(k) against k: where is it
# smallest, and what shape does it have on both sides? Why does k start at 2, not at 1?
st.predict("M20-T3", """

""")

In [ ]:
def aic_onset(data):
    """First sample of the second part of a short record, by the Akaike information criterion (Maeda 1985).

    data    : 1-D sequence of at least 5 numbers
    returns : int k, 2 <= k <= len(data) - 2, that minimizes k ln var(data[:k]) + (N - k - 1) ln var(data[k:])
    """
    # Step 1: a float64 array and its length N.
    # Step 2: the AIC for every split k from 2 to N - 2 (both parts keep at least two samples).
    # Step 3: the k of the smallest AIC (the first one on a tie), as a Python int.
    # YOUR CODE HERE
    raise NotImplementedError("M20-T3: write your code here")

In [ ]:
st.check("M20-T3", func=aic_onset)

Your function on a made-up record (signal-to-noise ratio 10), in windows of different lengths around the true
onset. The error is your onset minus the true onset, in samples (one sample is 0.01 s here).

In [ ]:
aic_trace, aic_truth = synth_onsets.onset_trace(np.random.default_rng(31), snr=10.0)
true_k = aic_truth["onset_index"]
for before_s, after_s in ((2.0, 0.5), (2.0, 5.0), (0.3, 0.5), (8.0, 0.5)):
    lo = true_k - int(before_s * 100)
    hi = true_k + int(after_s * 100)
    found = lo + aic_onset(aic_trace.data[lo:hi + 1])
    print(f"window from {before_s:3.1f} s before to {after_s:3.1f} s after the onset: "
          f"error {found - true_k:+4d} samples")

**Explain (3-5 sentences):** How close did your AIC onset get in the four windows? What goes wrong when the
window holds much more signal than noise, or almost no noise at all, and why? Why does a picker run STA/LTA
first and AIC only inside a short window around the trigger?

## Part 5 · An automatic picker on a filtered record (about 15 min, then task 4)

Real records need a **band-pass filter** (glossary `bandpass_filter`) before a picker runs: the slow ocean noise
near 0.2 Hz and high-frequency noise would otherwise dominate the ratio. M10 introduced the two kinds of filter:
a **causal filter** (glossary `causal_filter`) uses only past samples; a **zero-phase filter** (glossary
`zero_phase_filter`) runs the same filter forward and backward. Task 4 puts your three functions together into a
picker, and then you measure its picks on made-up records whose true onset you know, with both kinds of filter.

### M20-T4 · An automatic picker on a filtered record · core · about 75 min

**Goal:** write `auto_pick(tr, fmin, fmax, zerophase, sta_s=0.5, lta_s=10.0, thr_on=4.0, thr_off=1.5, pre_s=2.0,
post_s=0.5)`: band-pass a copy of a record, find the first trigger with your `sta_lta` and `trigger_windows`, and
time its onset with your `aic_onset`.

**Given:** your three functions and ObsPy's `Trace` methods (`copy`, `detrend`, `taper`, `filter`).

**Contract:**
1. Work on a copy of `tr`: remove the mean (`detrend("demean")`), taper 5 % at each end
   (`taper(max_percentage=0.05, type="hann")`), then band-pass `fmin`-`fmax` Hz with a Butterworth filter of
   4 corners (`filter("bandpass", ..., corners=4, zerophase=zerophase)`): zero-phase when `zerophase` is True,
   causal when it is False.
2. Window lengths in samples: `nsta = round(sta_s * sampling rate)`, `nlta = round(lta_s * sampling rate)`.
3. Your `sta_lta` on the band-passed samples, then your `trigger_windows(ratio, thr_on, thr_off)`. No trigger:
   return `None`.
4. Take the `on` sample `i` of the first trigger. The AIC window runs from `a = max(0, i - round(pre_s * rate))`
   to `b = min(N - 1, i + round(post_s * rate))`, both included (`N` samples in the record). The onset is
   `k = a + aic_onset(band-passed samples a to b)`.
5. Return the moment of sample `k`: `tr.stats.starttime + k * tr.stats.delta`, an `obspy.UTCDateTime`.
- Use your own three functions: `obspy.signal.trigger`, the `trigger` method of a trace or a stream, and the
  pyrocko package are not allowed here.

**Done when** `st.check("M20-T4", func=auto_pick)` passes: on made-up records (sharp and slow onsets, 50 to 200
samples per second, both kinds of filter, some records without any arrival), compared within 2 samples with the
same five steps done with ObsPy's own functions.

**Stuck?** `st.hint("M20-T4", 1)`

In [ ]:
# Before you code: after the check you will run auto_pick on made-up records with a sharp onset at a known time,
# band-passed 1-10 Hz once causal and once zero-phase, for signal-to-noise ratios of 5, 20 and 100. For each kind
# of filter: will the picks come before, at or after the true onset, by about how many samples (100 samples per
# second), and will the answer change with the signal-to-noise ratio?
st.predict("M20-T4", """

""")

In [ ]:
def auto_pick(tr, fmin, fmax, zerophase, sta_s=0.5, lta_s=10.0, thr_on=4.0, thr_off=1.5, pre_s=2.0, post_s=0.5):
    """Automatic onset: your STA/LTA trigger, then your AIC onset, on a band-passed copy of tr.

    tr               : ObsPy Trace (never changed)
    fmin, fmax       : band-pass corners (Hz), Butterworth with 4 corners
    zerophase        : True for a zero-phase band-pass, False for a causal one
    sta_s, lta_s     : STA and LTA window lengths (s)
    thr_on, thr_off  : trigger thresholds
    pre_s, post_s    : AIC window: seconds before and after the first trigger-on sample
    returns          : obspy UTCDateTime of the onset, or None when nothing triggers
    """
    # Step 1: a band-passed copy (mean removed, 5 % Hann taper, 4 corners, zero-phase or causal as asked).
    # Step 2: windows in samples; your ratio and your triggers; None when there is no trigger.
    # Step 3: the AIC window around the first on sample (clipped to the record); your AIC onset inside it.
    # Step 4: the moment of that sample.
    # YOUR CODE HERE
    raise NotImplementedError("M20-T4: write your code here")

In [ ]:
st.check("M20-T4", func=auto_pick)

Now the measurement. For each signal-to-noise ratio of `SNR_LEVELS`, the cell makes `N_MADE_UP` records with a
sharp onset at a known sample, picks each one with your `auto_pick` (causal, then zero-phase; band and settings
from the parameters cell) and prints the error, your pick minus the true onset, in samples: the median and the
10 % and 90 % points of its spread. It takes about half a minute.

In [ ]:
measured = []
examples = {}
for snr in SNR_LEVELS:
    made_up = synth_onsets.onset_set(np.random.default_rng(int(snr)), N_MADE_UP, snr=snr)
    for zerophase in (False, True):
        late = []
        for trace, truth in made_up:
            pick = auto_pick(trace, FMIN_HZ, FMAX_HZ, zerophase, STA_S, LTA_S, THR_ON, THR_OFF, AIC_PRE_S, AIC_POST_S)
            if pick is not None:
                late.append((pick - truth["onset"]) * trace.stats.sampling_rate)
        late = np.round(np.array(late))
        examples[(snr, zerophase)] = (made_up[0], auto_pick(made_up[0][0], FMIN_HZ, FMAX_HZ, zerophase, STA_S, LTA_S,
                                                             THR_ON, THR_OFF, AIC_PRE_S, AIC_POST_S))
        measured.append({"SNR": snr, "filter": "zero-phase" if zerophase else "causal",
                         "picked": f"{late.size}/{len(made_up)}", "median": np.median(late),
                         "10 %": np.percentile(late, 10), "90 %": np.percentile(late, 90)})
shift_table = pd.DataFrame(measured)
print("pick minus true onset, in samples (100 samples per second):")
print(shift_table.to_string(index=False))

(example_trace, example_truth), _ = examples[(SNR_LEVELS[-1], False)]
fig, ax = plt.subplots(figsize=(7.5, 3.4))
offset = 0.0
for zerophase, color in ((False, "tab:blue"), (True, "tab:red")):
    version = example_trace.copy().detrend("demean").taper(0.05, type="hann").filter(
        "bandpass", freqmin=FMIN_HZ, freqmax=FMAX_HZ, corners=4, zerophase=zerophase)
    t_rel = version.times() - example_truth["onset_s"]
    keep = (t_rel > -0.6) & (t_rel < 0.4)
    scale = np.abs(version.data[keep]).max()
    ax.plot(t_rel[keep], version.data[keep] / scale + offset, color=color, lw=0.8,
            label="zero-phase" if zerophase else "causal")
    pick = examples[(SNR_LEVELS[-1], zerophase)][1]
    if pick is not None:
        ax.plot(pick - example_truth["onset"], offset, "v", color=color, ms=7)
    offset += 2.2
ax.axvline(0.0, color="k", lw=0.8, ls="--")
ax.set_xlabel(f"Time after the true onset (s); SNR {SNR_LEVELS[-1]:g}; triangles: your picks")
ax.set_yticks([])
fig.legend(loc="upper center", ncol=2, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "picks_causal_zero_phase.png", dpi=150)

**Explain (3-5 sentences):** From your table: for each kind of filter, do the picks come early or late, by how
many samples, and how does that change with the signal-to-noise ratio? At which ratio does one kind look best,
and why is that misleading for a network, where near and far stations have very different ratios? Which kind of
filter will you use for picking, and what does your figure say about reading a first-motion polarity?

## Part 6 · Picking by hand (about 20 min, then task 5)

Now the real records. Your picks will be the reference for your picker (Part 7) and for the deep-learning
pickers of M21, and M31 will locate the earthquake with them, so take your time.

**Polarity and the instrument.** Your polarity column will say which way each record first moved. The counts move
the same way as the ground unless the channel records the ground upside down (a **polarity reversal**, glossary
`polarity_reversal`). M03 showed how the metadata would say so: the response of a channel is a chain of stages,
each with a **stage gain** (glossary `stage_gain`), and by the StationXML convention a negative product of the
stage gains marks a reversed channel. The file also stores one number for the whole chain, the **overall
sensitivity** (glossary `overall_sensitivity`). The cell below reads three signs for the vertical channel of every
station within `MAX_EPI_KM`: the sign of the first-stage gain (the sensor), the sign of the product of all stage
gains, and the sign of the stored sensitivity. The metadata are StationMetadata, integrated by Byeong-Seok Ahn
(Pukyong National University) and described by Ahn, Kang & Jung (2024); cite them whenever you use them.

In [ ]:
META_DIR = st.data.root() / "metadata" / "stationxml"


def sign_of(value):
    """'+' or '-' for a number."""
    return "+" if value > 0 else "-"


signs = []
for row in stations.loc[stations["epi_km"] <= MAX_EPI_KM].itertuples():
    line = {"station": row.station, "epi_km": round(row.epi_km, 1)}
    try:
        inventory = read_inventory(str(META_DIR / f"{row.station}.xml"))
        channel = inventory.select(channel=f"{BAND}Z", time=ORIGIN)[0][0][0]
        gains = [stage.stage_gain for stage in channel.response.response_stages]
        line.update({"first stage": sign_of(gains[0]), "all stages": sign_of(np.prod(gains)),
                     "stored sensitivity": sign_of(channel.response.instrument_sensitivity.value)})
    except Exception as exc:  # missing file, or no channel epoch at this time
        line.update({"first stage": f"unknown ({type(exc).__name__})", "all stages": "", "stored sensitivity": ""})
    signs.append(line)
signs = pd.DataFrame(signs)
print(f"Signs of the {BAND}Z response at the {len(signs)} stations within {MAX_EPI_KM:g} km (StationMetadata):")
print(signs.to_string(index=False))
odd = signs.loc[(signs["all stages"] == "-") & (signs["stored sensitivity"] == "+"), "station"]
print(f"\n{len(odd)} of {len(signs)} stations: the stages multiply to a negative number, but the stored sensitivity "
      f"is positive: {', '.join(odd) or 'none'}")

At several broadband stations the file disagrees with itself: the stage gains multiply to a negative number,
which by the convention means "reversed", while the stored sensitivity is positive, which means "not reversed".
M03 showed that in such a case the records decide, not the file, and that for this earthquake the counts are not
reversed: KS.SAJB and the other channels with a minus sign move first the same way as their neighbours at a
similar **azimuth** (glossary `azimuth`, the direction from the epicenter to a station, measured clockwise from
north), and the same way as KIGAM's published mechanism (the orientation of the fault and the direction of slip,
M51) predicts. Write every polarity as you see it in the counts: a U in your file means up in counts, which for
this earthquake is up in the ground. After task 5 you will compare your own polarities with those of their
neighbours.

**How to pick.** Use the picker E2 below. It shows the three components of one station; pick P on the vertical
(Z) record and S on a horizontal (N or E) record. Click (or type the time) to set a pick for the phase you
chose, set its uncertainty and, for P, its polarity, and press Save. It never picks for you.

- Zoom in. Set the window to a second or two around the onset, so that every sample shows, and compare the onset
  with the noise just before it.
- Filter only causally, or not at all (task 4 measured why; Scherbaum & Bouin 1997 discuss the same effect on
  real onsets). The picker's band-pass is causal; leave the zero-phase box off.
- P: put the pick at the last noise sample before the arrival leaves the noise. The polarity is the sign of the
  first swing away from the noise; if the noise is as large as that swing, write `?`.
- S: it arrives in the P wave's coda, so look for the change in size and in rhythm on both horizontals; its
  uncertainty is usually larger than the P's.
- Uncertainty: half the range of times you would still accept for the onset. For a sharp P in a quiet record that
  is a few hundredths of a second; for an emergent S, a tenth or more.
- Begin with the nearest stations and pick at least 6 stations within `MAX_EPI_KM` km, more if you can. You can
  stop and come back: the picker reads your file again when it starts.

### M20-T5 · Picking P and S by hand · core · about 120 min

**Goal:** pick the P and S arrivals of the ML 3.9 Sangju earthquake by hand at 6 or more stations within
`MAX_EPI_KM` (100) km of the epicenter, each with an uncertainty, and the first-motion polarity of every clear P;
save them to `PICKS_CSV` (`work/picks/M20_manual_picks.csv`).

**Given:** the HH records of your download, the station list of Part 0 and the picker E2, which writes the file.

**Contract (the file):**
- One row per pick, with the columns `station` (network.station, for example `XX.STA1`, an invented code),
  `phase` (`P` or `S`), `channel` (the channel you picked on: P on a vertical channel ending in Z, S on a
  horizontal channel ending in N or E), `time_utc` (ISO 8601 in UTC, for example `2031-04-05T06:07:08.090000Z`,
  an invented time), `uncertainty_s` (P: 0.005 to 0.5 s; S: 0.01 to 1.0 s) and `polarity` (P: `U`, `D` or `?`;
  S: `?`).
- At least 6 stations within `MAX_EPI_KM` km of the KMA epicenter, each with exactly one P and one S.
- At every station S comes after P, and each record must change at your picks.
- The S-P times of all stations must fit one earthquake: plotted against the P times, they must lie close to one
  straight line (S-P grows with distance at a steady rate).
- At least 3 P picks with a polarity `U` or `D`: the direction of the first swing away from the noise, in counts.

**Done when** `st.check("M20-T5", answer=PICKS_CSV, dataset=DATASET)` passes. The check reads your file and your
SAC files: the format, the stations and their distances, the channels, that the records change at your picks,
and that your S-P times fit one earthquake. It never compares your picks with hidden answers.

**Stuck?** `st.hint("M20-T5", 1)`

In [ ]:
# Before you pick: with the KMA hypocenter and a crust with VP about 6 km/s and VP/VS about 1.73, about when do you
# expect P and S at the nearest station and at a station about 55 km from the epicenter (seconds after the
# origin)? Which of the two phases will be harder to pick, and why?
st.predict("M20-T5", """

""")

In [ ]:
# The picker E2: one station at a time; your picks go to PICKS_CSV when you press Save.
picker = st.widgets.phase_picker(SAC_DIR / BAND, origin=ORIGIN, epicenter=(EV_LAT, EV_LON), max_km=MAX_EPI_KM,
                                 csv_path=PICKS_CSV, window=(-5.0, 30.0), fmin=FMIN_HZ, fmax=FMAX_HZ)
picker

In [ ]:
# Pick in the picker above and press Save. You can also type picks, one call per pick:
#     picker.set_pick("KS.XXXX", "P", <seconds after the origin>, <uncertainty in s>, "U", channel="HHZ")
# then save with picker.save(). Replace the two lines below with the calls you used (at least picker.save()).
# YOUR CODE HERE
raise NotImplementedError("M20-T5: write your code here")

In [ ]:
st.check("M20-T5", answer=PICKS_CSV, dataset=DATASET)

Your picks as a table, with the S-P time and the hypocentral distance of each station, and the S-P times drawn
against distance. For the nearest station the cell also prints the rough look of Part 1, the first sample above 20
times the noise, next to your P pick there.

In [ ]:
mine = pd.read_csv(PICKS_CSV)
both = mine.pivot_table(index="station", columns="phase", values="time_utc", aggfunc="first")
table = stations.set_index("station").loc[both.index, ["epi_km", "hypo_km"]].copy()
table["P_s"] = [UTCDateTime(t) - ORIGIN for t in both["P"]]
table["S_s"] = [UTCDateTime(t) - ORIGIN for t in both["S"]]
table["S-P_s"] = table["S_s"] - table["P_s"]
table = table.sort_values("hypo_km")
print(table.round(3).to_string())
nearest_p = f"your P pick {table.loc[NEAREST, 'P_s']:.2f} s" if NEAREST in table.index else "not picked"
print(f"\n{NEAREST} (nearest station): first sample above 20 times the noise {first_big:.2f} s after the origin; "
      f"{nearest_p}")
fig, ax = plt.subplots(figsize=(5.5, 4.0))
ax.plot(table["hypo_km"], table["S-P_s"], "o", color="k", ms=5)
for name, row in table.iterrows():
    ax.annotate(name.split(".")[1], (row["hypo_km"], row["S-P_s"]), xytext=(4, -3), textcoords="offset points",
                fontsize=7)
ax.set_xlabel("Hypocentral distance from the KMA hypocenter (km)")
ax.set_ylabel("S-P time (s)")
fig.tight_layout()
fig.savefig(Path(FIGURE_DIR) / "manual_s_minus_p.png", dpi=150)

Your polarities next to the metadata of Part 6, sorted by azimuth. The first motion of P depends on the direction
in which the wave left the source: in some directions the ground is first pushed away from it (U), in others
pulled toward it (D); M51 maps this. Stations in nearly the same direction from the epicenter therefore usually
move first the same way, unless the boundary between push and pull passes between them. A channel whose counts
ran upside down would stand out against its neighbours.

In [ ]:
first_gain = signs.set_index("station")["first stage"]
p_polarity = mine.loc[mine["phase"] == "P"].set_index("station")["polarity"]
facing = []
for name in p_polarity.index:
    head = read(str(stations.set_index("station").loc[name, "z_file"]), headonly=True)[0].stats
    _, azimuth_deg, _ = gps2dist_azimuth(EV_LAT, EV_LON, head.sac.stla, head.sac.stlo)
    facing.append({"station": name, "azimuth_deg": round(azimuth_deg), "epi_km": round(table.loc[name, "epi_km"], 1),
                   "your polarity": p_polarity[name], "first-stage gain": first_gain.get(name, "unknown")})
facing = pd.DataFrame(facing).sort_values("azimuth_deg", ignore_index=True)
print(facing.to_string(index=False))

**Explain (3-5 sentences):** Which pick was hardest, and how did you choose your uncertainties? Does your S-P time
grow steadily with distance, and which station stands out? In the azimuth table, do the stations whose first-stage
gain is negative move first like their neighbours at a similar azimuth? Your own picks may give few such
neighbours: say which stations you could test and which you could not (in M21, PocketQuake picks many more
stations, and you can come back to this question there).

## Part 7 · Your picker against your own picks (about 20 min, then task 6)

Your manual picks are an independent reference for your picker: you made them with your eyes, not with its
rules. Comparing the two on the same records shows how well the picker works on real data and which settings
suit these records. A few large errors (a trigger on noise, a missed station) should not hide how well the rest
agree, so use **robust statistics**: the median error and the **median absolute error** (glossary
`median_absolute_error`, the median of the error sizes without their signs), not the mean.

### M20-T6 · Your picker against your own picks · core · about 90 min

**Goal:** run your `auto_pick` on the vertical records of the stations you picked by hand, for a small grid of
settings; measure its errors against your manual P picks; choose one setting and save the result.

**Given:** your `auto_pick` and your picks file.

**Contract:**
- Records: the `HHZ` record of each station of your file, cut to `PICK_WINDOW_S` around the KMA origin:
  `read(path)[0].trim(ORIGIN + PICK_WINDOW_S[0], ORIGIN + PICK_WINDOW_S[1])`.
- Picker: your `auto_pick` with `FMIN_HZ`-`FMAX_HZ`, causal (`zerophase=False`), `pre_s = AIC_PRE_S`,
  `post_s = AIC_POST_S`.
- Grid: at least two values of `sta_s` and two of `thr_on` (4 settings or more); `lta_s` and `thr_off` as you like.
- Error at a station: your automatic pick minus your manual P pick, in seconds (positive: the picker is late). A
  station without a trigger is missed and has no error.
- For every setting: `n_picked` (stations with an automatic pick), `median_error_s`, `median_abs_error_s` and
  `max_abs_error_s` (the median of the errors, the median of their absolute values and the largest absolute
  value, over the picked stations).
- Write a JSON file with the keys `picks_csv` (the path of your picks file), `band_hz` (`[fmin, fmax]`),
  `window_s` (`PICK_WINDOW_S`), `aic_s` (`[pre_s, post_s]`), `grid` (a list with one dict per setting: `sta_s`,
  `lta_s`, `thr_on`, `thr_off`, `n_picked`, `median_error_s`, `median_abs_error_s`, `max_abs_error_s`), `chosen`
  (the grid entry you choose) and `rows` (one dict per station of your file for the chosen setting: `station`,
  `sac_path`, `manual_p_utc`, `auto_p_utc` and `error_s`, the last two `null` when missed). Times as ISO text in
  UTC.
- Set `auto_result = {"json": <path of the JSON file>, "picker": auto_pick}`.

**Done when** `st.check("M20-T6", answer=auto_result, dataset=DATASET)` passes. The check first runs your
`auto_pick` on made-up records of task 4 again; then it reads your JSON file and your picks file, runs your
`auto_pick` itself on your records with your chosen setting and with one more setting of your grid, and
recomputes your errors and statistics. It never prints its own values.

**Stuck?** `st.hint("M20-T6", 1)`

In [ ]:
# Before you code: with the start settings (STA 0.5 s, LTA 10 s, thr_on 4, thr_off 1.5, causal 1-10 Hz), at how
# many of your stations will the first trigger be the P wave? What else could make the first trigger, and how large
# do you expect the errors of the good picks to be, after what you measured in task 4?
st.predict("M20-T6", """

""")

In [ ]:
# Read your picks and the HHZ records; run auto_pick for every setting of your grid; compute the errors and the
# statistics; choose a setting; write the JSON file; set auto_result.
# YOUR CODE HERE
raise NotImplementedError("M20-T6: write your code here")

In [ ]:
st.check("M20-T6", answer=auto_result, dataset=DATASET)

**Explain (3-5 sentences):** Which setting did you choose, and what in your grid table supports that choice? Which
settings went wrong, how do the median and the largest error show it, and where did their first triggers fall
(print the rows of one of them)? How does the median error of your chosen setting compare with the causal bias you
measured on made-up records in task 4, and what does that say about your manual picks?

## Part 8 · A production picker: unchanged, then one change (about 30 min)

ObsPy ships ready-made pickers. `pk_baer` is the picker of Baer & Kradolfer (1987), used at many observatories.
Its characteristic function adds, at every sample, the squared value of the record and the squared rate of change
of the record (how fast it changes from one sample to the next), weighted so that the two count about equally,
and squares that sum again so that arrivals stand out. It measures this function in standard deviations above
its mean in the noise. The pick is the first sample of the first trigger that lasts long enough; no onset timing
follows, unlike the AIC step of your `auto_pick`. Each pick comes with a short code: `I` (impulsive) or `E`
(emergent), then `P`, then `U` when the record rises at the pick or `D` when it falls, and a quality digit from 0
(best) to 4. It has six settings, two thresholds and four lengths in samples (here 100 samples are 1 s); ObsPy's
documentation and its source code (`pk_mbaer.c`) use them like this:

- `preset_len`: sets the picker up at the start of the record. The weight between the record and its rate of
  change comes from the first `preset_len` samples, and the first noise level from the first 2 × `preset_len`
  samples, in which no trigger can start.
- `thr1`: the trigger threshold. A trigger starts when the function rises above `thr1` standard deviations of the
  noise.
- `thr2`: the noise threshold. The mean and the standard deviation of the noise are updated with every sample
  whose function stays below `thr2`, so that a strong arrival does not count as noise.
- `tdownmax`: how many samples the function may drop below `thr1` without ending the trigger. A longer drop ends
  it, and the picker then decides whether the trigger lasted long enough.
- `tupevent`: how long a trigger must have lasted, in samples, to become the P pick (samples above 4 × `thr1`
  count extra). A shorter trigger is dropped, and the search goes on.
- `p_dur`: the samples after the trigger in which the largest amplitude is measured. That amplitude, divided by
  the largest amplitude before the trigger, sets the quality digit: 0 above 8, 1 above 6, 2 above 4, 3 above 1.5
  and 4 otherwise; `I` goes with quality 0 or 1, `E` with 2 to 4.

A production tool must be tested like your own code. Three runs on the same cut records as task 6 (as recorded,
mean removed), each compared with your manual P picks: first exactly as ObsPy's documentation shows
(`tdownmax = 20`, `tupevent = 60`, `thr1 = 7.0`, `thr2 = 12.0`, `preset_len = 100`, `p_dur = 100`); then again,
unchanged; then with the one setting of `BAER_CHANGE` (parameters cell) changed.

In [ ]:
from obspy.signal.trigger import pk_baer

BAER_SETTINGS = {"tdownmax": 20, "tupevent": 60, "thr1": 7.0, "thr2": 12.0, "preset_len": 100, "p_dur": 100}
p_rows = mine.loc[mine["phase"] == "P"].set_index("station")
cut_records = {}
for name in p_rows.index:
    z_path = next((SAC_DIR / BAND).glob(f"*.{name}.{BAND}Z.sac"))
    record = read(str(z_path))[0].trim(ORIGIN + PICK_WINDOW_S[0], ORIGIN + PICK_WINDOW_S[1])
    cut_records[name] = record.detrend("demean")


def run_baer(settings):
    """pk_baer with these settings on every cut record: its pick minus your P pick (s), and its code."""
    lines = []
    for name, record in cut_records.items():
        sample, code = pk_baer(record.data, record.stats.sampling_rate, **settings)
        picked = record.stats.starttime + sample * record.stats.delta
        error = round(float(picked - UTCDateTime(p_rows.loc[name, "time_utc"])), 3) if code else None
        lines.append({"station": name, "your polarity": p_rows.loc[name, "polarity"], "error_s": error,
                      "code": code or "no pick"})
    return pd.DataFrame(lines)


def agreement(result):
    """How many picks lie within 0.1 s of yours, and at how many of those the U/D letter is your polarity."""
    close = pd.to_numeric(result["error_s"]).abs() <= 0.1
    same = close & (result["code"].str[2] == result["your polarity"])
    return (f"{int(close.sum())} of {len(result)} picks within 0.1 s of yours; "
            f"the U/D letter agrees with your polarity at {int(same.sum())} of them")


baer_first = run_baer(BAER_SETTINGS)
print(baer_first.to_string(index=False))
print(agreement(baer_first))

**An unchanged rerun.** Is `pk_baer` **deterministic** (glossary `deterministic_computation`): does it give exactly
the same result when nothing has changed? If not, every later comparison mixes your change with that scatter.

In [ ]:
baer_rerun = run_baer(BAER_SETTINGS)
print("rerun identical to the first run:", baer_rerun.equals(baer_first))

**One setting changed.** `BAER_CHANGE` in the parameters cell names one setting and its new value. To try another
change later, edit it there, run the parameters cell again, and then this cell.

In [ ]:
changed_name, changed_value = BAER_CHANGE
if changed_name not in BAER_SETTINGS:
    raise KeyError(f"BAER_CHANGE must name one of {list(BAER_SETTINGS)}, not {changed_name!r}")
baer_changed = run_baer(dict(BAER_SETTINGS, **{changed_name: changed_value}))
print(f"changed: {changed_name} from {BAER_SETTINGS[changed_name]} to {changed_value}; all other settings unchanged")
side_by_side = baer_first.merge(baer_changed, on=["station", "your polarity"], suffixes=(" unchanged", " changed"))
print(side_by_side.to_string(index=False))
print("unchanged:", agreement(baer_first))
print("changed:  ", agreement(baer_changed))

**Explain (3-5 sentences, not checked):** At how many stations did the unchanged `pk_baer` agree with your P pick
within 0.1 s, and does its U/D letter agree with your polarity there? Where it failed, what did it pick instead,
and which of its settings let that happen? Was the rerun identical, and why does that matter before you change
anything? What did your one change fix or break, and would you keep it for a weaker earthquake?

## Wrap-up

Your progress in this module (from `work/.sgtl/progress.jsonl`):

In [ ]:
st.progress("M20")

### Going further (optional, not checked)

1. Run your `auto_pick` on the horizontal records of your stations, with the window starting 1 s after your P
   pick. How often does it find your S pick, and how late is it?
2. Make the made-up onsets emergent (`synth_onsets.onset_set(rng, 40, snr=20.0, rise_s=0.3)`) and repeat the
   measurement of Part 5. How much later are the causal picks, and why?
3. Repeat Part 5 with a band of 2-15 Hz and with 8 corners instead of 4. Which way does the zero-phase shift
   move, and why?
4. Pick the same stations on the HG (accelerometer) records of your download. Do your P picks and polarities
   agree with your HH picks?

Next: M21 runs deep-learning pickers on these records and compares their picks and polarities with yours.